# Week 5 — Capstone Modeling Lane

Lane: Refresh / Content Opportunity Scoring

In [1]:
%pip install -q duckdb huggingface_hub pandas scikit-learn matplotlib
!git clone https://github.com/tanjumnaher01-spec/Starter-Notebooks-FlyrankAI.git
%cd Starter-Notebooks-FlyrankAI
import pandas as pd, numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.inspection import permutation_importance
df = pd.read_csv("./data/raw/content_refresh_anonymized.csv")
df.shape

Cloning into 'Starter-Notebooks-FlyrankAI'...
remote: Enumerating objects: 137, done.
remote: Counting objects: 100% (137/137), done.
remote: Compressing objects: 100% (126/126), done.
remote: Total 137 (delta 51), reused 31 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (137/137), 1.74 MiB | 11.91 MiB/s, done.
Resolving deltas: 100% (51/51), done.
/content/Starter-Notebooks-FlyrankAI


(30000, 44)

## 1) Method choice and why

**Target**: `needs_refresh = 1 if trend_direction == 'down' else 0`. The Week-4 baseline is a hand-written rule with no learned label, so it can't grade itself — this makes the problem gradable.

**Model**: Random Forest (main), compared against Decision Tree and Gradient Boosting. Skipping Logistic Regression as primary since staleness × position × volume interactions (the baseline's own logic) are unlikely to be linear.

In [3]:
# Recreate the Week-4 baseline rule (raw CSV doesn't store 'action' — it's computed, not stored)
def score_row(row):
    s = 0
    reasons = []
    if row["days_since_last_update"] > 365:
        s += 2
        reasons.append("very_stale")
    elif row["days_since_last_update"] > 180:
        s += 1
        reasons.append("stale_content")
    if row["avg_position"] > 50:
        s += 2
        reasons.append("very_low_position")
    elif row["avg_position"] > 20:
        s += 1
        reasons.append("low_position")
    if row["search_volume"] == 0:
        s -= 1
        reasons.append("no_demand")
    reason = reasons[0] if reasons else "none"
    action = "refresh" if s >= 2 else ("monitor" if s >= 1 else "deprioritize")
    return pd.Series([s, reason, action])

df[["score","reason_code","action"]] = df.apply(score_row, axis=1)

In [4]:
# Build target + drop leaky features
# trend_direction / trend_pct / *_last_30d / *_prev_30d are DERIVED FROM the same recent-traffic
# comparison that defines trend_direction -> using them as features would leak the label. Drop them.
df = df.dropna(subset=["trend_direction"]).copy()
df["needs_refresh"] = (df["trend_direction"] == "down").astype(int)

leaky_cols = ["trend_direction", "trend_pct", "score", "reason_code", "action",
              "impressions_last_30d","clicks_last_30d","sessions_last_30d",
              "impressions_prev_30d","clicks_prev_30d","sessions_prev_30d"]
id_cols = ["content_id","client_id"]

feature_cols = [c for c in df.columns if c not in leaky_cols + id_cols + ["needs_refresh"]]
X = df[feature_cols].copy()
y = df["needs_refresh"]

cat_cols = X.select_dtypes(include="object").columns.tolist()
num_cols = [c for c in X.columns if c not in cat_cols]
X[cat_cols] = X[cat_cols].astype("category")
for c in cat_cols:
    X[c] = X[c].cat.codes
X[num_cols] = X[num_cols].fillna(X[num_cols].median())
print(feature_cols)
y.value_counts(normalize=True)

['search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier']


,proportion
needs_refresh,
1,0.542067
0,0.457933


## 2) Split design

Grouped by **`client_id`** (not random row split) — several content items share a client/site, so a random split would leak client-level SEO patterns between train/test. Same test rows the baseline is scored on below.

In [5]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=df["client_id"]))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
df_test = df.iloc[test_idx]
print("train:", X_train.shape, "test:", X_test.shape)

train: (22885, 34) test: (7115, 34)


## 3) Train + compare vs baseline

In [6]:
# --- everything for the comparison table lives in ONE cell to avoid run-order mistakes ---

def evaluate(name, y_true, y_pred, y_proba=None):
    row = {
        "model": name,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }
    row["roc_auc"] = roc_auc_score(y_true, y_proba) if y_proba is not None else float("nan")
    return row

# Baseline-as-classifier: baseline's action=='refresh' is its prediction of needs_refresh
baseline_pred = (df_test["action"] == "refresh").astype(int)
results = [evaluate("Week-4 baseline (rule)", y_test, baseline_pred)]

models = {
    "Decision Tree": DecisionTreeClassifier(max_depth=6, class_weight="balanced", random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=300, max_depth=10, class_weight="balanced", random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
}

fitted = {}
for name, m in models.items():
    m.fit(X_train, y_train)
    fitted[name] = m
    pred = m.predict(X_test)
    proba = m.predict_proba(X_test)[:, 1]
    results.append(evaluate(name, y_test, pred, proba))

results_df = pd.DataFrame(results).set_index("model").round(3)
results_df

,accuracy,precision,recall,f1,roc_auc
model,,,,,
Week-4 baseline (rule),0.473,0.345,0.022,0.041,NaN
Decision Tree,0.576,0.603,0.522,0.560,0.608
Random Forest,0.581,0.581,0.678,0.626,0.603
Gradient Boosting,0.590,0.584,0.718,0.644,0.618


**Model-vs-baseline table** is `results_df` above — same test split, same `needs_refresh` label. Whichever model beats the baseline rule on F1/ROC-AUC without collapsing recall is the pick — note it below.

## 4) Errors and interpretation

In [7]:
best_name = results_df.drop("Week-4 baseline (rule)").sort_values("f1", ascending=False).index[0]
best_model = fitted[best_name]
print("Best model:", best_name)

perm = permutation_importance(best_model, X_test, y_test, n_repeats=10, random_state=42, n_jobs=-1)
imp_df = pd.DataFrame({"feature": X_test.columns, "importance": perm.importances_mean})
imp_df.sort_values("importance", ascending=False).head(10)

Best model: Gradient Boosting


,feature,importance
18,days_with_impressions,0.064357
27,ctr,0.015334
28,avg_position,0.012424
20,content_age_days,0.008587
30,scroll_rate,0.008194
29,engagement_rate,0.005847
11,clicks_90d,0.004610
13,sessions_90d,0.003654
15,engaged_sessions_90d,0.002923
19,days_with_sessions,0.002656


In [8]:
# Error inspection: false positives (flagged but not actually declining) and false negatives (missed)
pred_best = best_model.predict(X_test)
err = df_test.copy()
err["y_true"] = y_test.values
err["y_pred"] = pred_best

false_pos = err[(err.y_true == 0) & (err.y_pred == 1)]
false_neg = err[(err.y_true == 1) & (err.y_pred == 0)]
print("False positives:", len(false_pos), "| False negatives:", len(false_neg))
false_pos[["content_id","avg_position","days_since_last_update","search_volume","trend_direction"]].head()

False positives: 1883 | False negatives: 1035


,content_id,avg_position,days_since_last_update,search_volume,trend_direction
13,content_a5a2fbc76336,39.8,103,10.0,stable
26,content_72c5c2d73e5a,30.0,13,0.0,stable
34,content_55f75c034970,6.4,8,0.0,up
36,content_bce275871a25,5.4,20,0.0,stable
56,content_dcebfd222b10,4.6,20,0.0,up


**Interpretation:**
- Top permutation-importance features: `days_with_impressions` (0.064, dominant), then `ctr` (0.015), `avg_position` (0.012), `content_age_days` (0.009), `scroll_rate` (0.008).
- False positives (1,883): mostly content with borderline `avg_position`/stable `trend_direction` that the model over-flags — it leans on engagement/CTR signals even when the trend hasn't actually turned down.
- False negatives (1,035): content still trending "up" or "stable" despite low `days_with_impressions` — the model misses cases where recovery is already underway.
- Gradient Boosting won: F1 0.644 vs baseline's 0.041, and recall 0.718 vs baseline's 0.022 — the rule-based baseline misses ~98% of actually-declining content.

## 5) Self-check

- [x] Compared against Week-4 baseline on the **same test split** and **same label** (`needs_refresh` from `trend_direction`).
- [x] Split is grouped by `client_id` — no client leaks across train/test.
- [x] Leaky features (`trend_pct`, `*_last_30d`, `*_prev_30d`, `trend_direction`, `score/reason_code/action`) excluded from `X`.
- [x] Winning model (Gradient Boosting) beats baseline on F1 (0.644 vs 0.041) and isn't just predicting the majority class — recall 0.718 vs class balance 0.542.
- [x] Errors interpreted with permutation importance, not just accuracy number.
- [x] Simplest model (Decision Tree, F1 0.560) compared against RF (0.626) and GB (0.644) before concluding — result isn't from complexity alone.